Exploratory Data Analysis (EDA)
Данные о 1000 химических соединений: IC50, CC50, SI и молекулярные дескрипторы.

In [1]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

## 1. Загрузка данных

In [2]:
df = pd.read_excel("data/data.xlsx", index_col=0)
print("=" * 60)
print(f"Размер датасета: {df.shape[0]} строк × {df.shape[1]} столбцов")
print("=" * 60)

targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]
print(f"Целевые переменные: {targets}")
print(f"Признаки: {len(features)} штук\n")

Размер датасета: 1001 строк × 213 столбцов
Целевые переменные: ['IC50, mM', 'CC50, mM', 'SI']
Признаки: 210 штук



## 2. Пропуски и типы

In [3]:
print("Пропуски:", df.isnull().sum().sum())
print("\nТипы столбцов:")
print(df.dtypes.value_counts())

Пропуски: 36

Типы столбцов:
float64    107
int64      106
Name: count, dtype: int64


## 3. Описательная статистика таргетов

In [4]:
print("\n── Описательная статистика целевых переменных ──")
print(df[targets].describe().T.to_string())


── Описательная статистика целевых переменных ──
           count        mean         std       min        25%         50%         75%           max
IC50, mM  1001.0  222.805156  402.169734  0.003517  12.515396   46.585183  224.975928   4128.529377
CC50, mM  1001.0  589.110728  642.867508  0.700808  99.999036  411.039342  894.089176   4538.976189
SI        1001.0   72.508823  684.482739  0.011489   1.433333    3.846154   16.566667  15620.600000


## 4. Корреляция IC50 и CC50 → SI = CC50/IC50

In [5]:
si_check = df["CC50, mM"] / df["IC50, mM"]
corr_si = np.corrcoef(si_check, df["SI"])[0, 1]
print(f"\nПроверка SI = CC50/IC50: корреляция = {corr_si:.6f}")


Проверка SI = CC50/IC50: корреляция = 1.000000


## 5. Распределения таргетов

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Распределения целевых переменных (оригинал и log-трансформация)", fontsize=14)

for i, col in enumerate(targets):
    # исходное
    axes[0, i].hist(df[col], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    axes[0, i].set_title(col)
    axes[0, i].set_xlabel("Значение")
    axes[0, i].set_ylabel("Частота")

    # log
    log_vals = np.log1p(df[col])
    axes[1, i].hist(log_vals, bins=50, color="tomato", edgecolor="white", alpha=0.8)
    axes[1, i].set_title(f"log1p({col})")
    axes[1, i].set_xlabel("log-значение")
    axes[1, i].set_ylabel("Частота")

plt.tight_layout()
plt.savefig("plots/eda_target_distributions.png")
plt.close()
print("\nСохранён график: eda_target_distributions.png")


Сохранён график: eda_target_distributions.png


## 6. Выбросы (IQR)

In [7]:
print("\n── Выбросы (IQR-метод) ──")
for col in targets:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"  {col}: {n_out} выбросов ({n_out/len(df)*100:.1f}%)")


── Выбросы (IQR-метод) ──
  IC50, mM: 147 выбросов (14.7%)
  CC50, mM: 39 выбросов (3.9%)
  SI: 125 выбросов (12.5%)


## 7. Корреляционная матрица таргетов

In [8]:
fig, ax = plt.subplots(figsize=(5, 4))
corr = df[targets].corr()
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", ax=ax, vmin=-1, vmax=1)
ax.set_title("Корреляция целевых переменных")
plt.tight_layout()
plt.savefig("plots/eda_target_correlation.png")
plt.close()
print("Сохранён график: eda_target_correlation.png")
print("\nКорреляции между таргетами:")
print(corr.to_string())

Сохранён график: eda_target_correlation.png

Корреляции между таргетами:
          IC50, mM  CC50, mM        SI
IC50, mM  1.000000  0.521346 -0.056604
CC50, mM  0.521346  1.000000 -0.006818
SI       -0.056604 -0.006818  1.000000


## 8. Топ-20 признаков по корреляции с каждым таргетом

In [9]:
print("\n── Топ-10 признаков по |корреляции| с IC50 ──")
corr_ic50 = df[features].corrwith(df["IC50, mM"]).abs().sort_values(ascending=False)
print(corr_ic50.head(10).to_string())

print("\n── Топ-10 признаков по |корреляции| с CC50 ──")
corr_cc50 = df[features].corrwith(df["CC50, mM"]).abs().sort_values(ascending=False)
print(corr_cc50.head(10).to_string())

print("\n── Топ-10 признаков по |корреляции| с SI ──")
corr_si_feat = df[features].corrwith(df["SI"]).abs().sort_values(ascending=False)
print(corr_si_feat.head(10).to_string())


── Топ-10 признаков по |корреляции| с IC50 ──
VSA_EState4     0.274203
Chi2n           0.257058
PEOE_VSA7       0.255988
Chi2v           0.249164
fr_Ar_NH        0.245511
fr_Nhpyrrole    0.245511
Chi4v           0.243600
Chi4n           0.243497
Chi3n           0.239741
Chi3v           0.237759

── Топ-10 признаков по |корреляции| с CC50 ──
MolMR             0.310111
LabuteASA         0.309191
MolWt             0.306439
ExactMolWt        0.306382
HeavyAtomCount    0.305169
Chi0              0.304792
Chi1              0.304380
HeavyAtomMolWt    0.303163
Kappa1            0.302206
Chi1v             0.301525

── Топ-10 признаков по |корреляции| с SI ──
BalabanJ            0.162955
fr_NH2              0.160470
RingCount           0.124444
fr_Al_COO           0.102414
fr_COO              0.101115
fr_COO2             0.101115
NumAromaticRings    0.088064
VSA_EState4         0.087837
FpDensityMorgan1    0.087341
VSA_EState6         0.082995


## 9. Heatmap топ-20 признаков → IC50

In [10]:
top20 = corr_ic50.head(20).index.tolist()
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    df[top20].corr(),
    cmap="coolwarm",
    ax=ax,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
)
ax.set_title("Корреляции топ-20 признаков (по связи с IC50)")
plt.tight_layout()
plt.savefig("plots/eda_feature_heatmap.png")
plt.close()
print("Сохранён график: eda_feature_heatmap.png")

Сохранён график: eda_feature_heatmap.png


## 10. Scatter: топ-3 признака vs IC50

In [11]:
top3 = corr_ic50.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Топ-3 признака vs IC50")
for i, feat in enumerate(top3):
    axes[i].scatter(df[feat], np.log1p(df["IC50, mM"]), alpha=0.3, s=10, color="steelblue")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("log1p(IC50)")
    r, p = stats.pearsonr(df[feat], df["IC50, mM"])
    axes[i].set_title(f"r = {r:.3f}")
plt.tight_layout()
plt.savefig("plots/eda_top_features_ic50.png")
plt.close()
print("Сохранён график: eda_top_features_ic50.png")

Сохранён график: eda_top_features_ic50.png


## 11. Нулевая дисперсия — удаляем

In [12]:
zero_var = [c for c in features if df[c].nunique() <= 1]
print(f"\nПризнаки с нулевой/единичной дисперсией ({len(zero_var)}): {zero_var[:10]}")


Признаки с нулевой/единичной дисперсией (18): ['NumRadicalElectrons', 'SMR_VSA8', 'SlogP_VSA9', 'fr_N_O', 'fr_SH', 'fr_azide', 'fr_barbitur', 'fr_benzodiazepine', 'fr_diazo', 'fr_dihydropyridine']


## 12. Классификационные метки

In [13]:
ic50_med = df["IC50, mM"].median()
cc50_med = df["CC50, mM"].median()
si_med = df["SI"].median()

print(f"\nМедианы:")
print(f"  IC50 = {ic50_med:.4f} mM")
print(f"  CC50 = {cc50_med:.4f} mM")
print(f"  SI   = {si_med:.4f}")
print(f"  SI > 8: {(df['SI'] > 8).sum()} соединений из {len(df)}")


Медианы:
  IC50 = 46.5852 mM
  CC50 = 411.0393 mM
  SI   = 3.8462
  SI > 8: 357 соединений из 1001


## 13. Boxplot таргетов

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Boxplot целевых переменных (log-шкала)")
for i, col in enumerate(targets):
    axes[i].boxplot(np.log1p(df[col]), patch_artist=True,
                    boxprops=dict(facecolor="lightblue"))
    axes[i].set_title(f"log1p({col})")
    axes[i].set_ylabel("log-значение")
plt.tight_layout()
plt.savefig("plots/eda_boxplots.png")
plt.close()
print("Сохранён график: eda_boxplots.png")

Сохранён график: eda_boxplots.png


## 14. Попарные scatter таргетов

In [15]:
fig = sns.pairplot(np.log1p(df[targets]), diag_kind="kde", plot_kws={"alpha": 0.3, "s": 10})
fig.fig.suptitle("Попарные зависимости log-таргетов", y=1.02)
plt.savefig("plots/eda_pairplot_targets.png")
plt.close()
print("Сохранён график: eda_pairplot_targets.png")

Сохранён график: eda_pairplot_targets.png


## 15. Итоговые выводы

- Датасет размером 1001 строк × 213 столбцов с 210 признаками
- Целевые переменные: ['IC50, mM', 'CC50, mM', 'SI']
- IC50 и CC50 умеренно коррелируют (0.52): эффективные соединения часто токсичны.
- Селективность (SI) не зависит от IC50 и CC50 по отдельности.
- Только у 357 из 1001 соединения SI > 8 (хорошая селективность).
- На эффективность (IC50) влияют азотсодержащие фрагменты (fr_Ar_NH, fr_Nhpyrrole) и топологические индексы (Chi2n, Chi2v).
- На токсичность (CC50) сильнее всего влияют размер молекулы: масса, площадь поверхности, число тяжелых атомов.
- На селективность (SI) связи слабые, но заметны BalabanJ, наличие аминов (fr_NH2), число колец и карбоксильные группы.
- 18 признаков имеют нулевую или единичную дисперсию (например, NumRadicalElectrons, fr_SH, fr_azide) — их можно удалить.